# Solutions — Capstone: Wanderbricks Platform (Hard 18)

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

bookings   = spark.table("samples.wanderbricks.bookings")
users      = spark.table("samples.wanderbricks.users")
properties = spark.table("samples.wanderbricks.properties")
reviews    = spark.table("samples.wanderbricks.reviews")
payments   = spark.table("samples.wanderbricks.payments")
hosts      = spark.table("samples.wanderbricks.hosts")


## Solution 1 — Active Host Portfolio

In [ ]:
# Rank hosts by count of active listings; collect property types
w = Window.orderBy(F.col("property_count").desc())

result_1 = (
    properties
    .filter(F.col("is_active") == True)
    .groupBy("host_id")
    .agg(
        F.count("*").alias("property_count"),
        F.collect_set("property_type").alias("property_types"),
    )
    .join(hosts.select("host_id", "name"), on="host_id")
    .withColumn("portfolio_rank", F.rank().over(w))
    .select("host_id", "name", "property_count", "property_types", "portfolio_rank")
    .orderBy("portfolio_rank")
)


## Solution 2 — Booking Revenue by Property Type and Country

In [ ]:
# Confirmed bookings only; per (property_type, country)
result_2 = (
    bookings
    .filter(F.col("status") == "confirmed")
    .join(properties.select("property_id", "property_type", "country"), on="property_id")
    .groupBy("property_type", "country")
    .agg(
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.avg(F.datediff("check_out", "check_in")), 2).alias("avg_stay_nights"),
    )
    .orderBy(F.col("total_revenue").desc())
)


## Solution 3 — Guest Spending Behaviour

In [ ]:
# Top guests by total spend; include countries visited (collect_set)
result_3 = (
    bookings
    .filter(F.col("status") == "confirmed")
    .join(users.select("user_id", "name"), on="user_id")
    .join(properties.select("property_id", "country"), on="property_id")
    .groupBy("user_id", "name")
    .agg(
        F.round(F.sum("total_amount"), 2).alias("total_spend"),
        F.collect_set("country").alias("countries_visited"),
    )
    .orderBy(F.col("total_spend").desc())
)


## Solution 4 — Review Quality by Property Type

In [ ]:
result_4 = (
    reviews
    .filter(F.col("is_deleted") == False)
    .join(properties.select("property_id", "property_type"), on="property_id")
    .groupBy("property_type")
    .agg(F.round(F.avg("rating"), 2).alias("avg_rating"))
    .withColumn("quality_flag",
        F.when(F.col("avg_rating") >= 4.5, "excellent")
         .when(F.col("avg_rating") >= 4.0, "good")
         .otherwise("needs_improvement")
    )
    .orderBy(F.col("avg_rating").desc())
)


## Solution 5 — Cancellation Intelligence

In [ ]:
# Per (property_type, country): cancellation rate and total bookings
result_5 = (
    bookings
    .join(users.select("user_id", F.col("country").alias("user_country")), on="user_id")
    .join(properties.select("property_id", "property_type"), on="property_id")
    .groupBy("property_type", "user_country")
    .agg(
        F.count("*").alias("total_bookings"),
        F.sum(F.when(F.col("status") == "cancelled", 1).otherwise(0)).alias("cancelled_bookings"),
    )
    .withColumn("cancellation_rate_pct",
        F.round(F.col("cancelled_bookings") / F.col("total_bookings") * 100, 2)
    )
    .orderBy(F.col("cancellation_rate_pct").desc())
)


## Solution 6 — Seasonal Booking Patterns

In [ ]:
# Per (property_type, month): booking count; collect_set peak months (top 3)
w_rank = Window.partitionBy("property_type").orderBy(F.col("booking_count").desc())

monthly = (
    bookings
    .join(properties.select("property_id", "property_type"), on="property_id")
    .withColumn("month", F.month("check_in"))
    .groupBy("property_type", "month")
    .agg(F.count("*").alias("booking_count"))
    .withColumn("month_rank", F.rank().over(w_rank))
)

result_6 = (
    monthly
    .filter(F.col("month_rank") <= 3)
    .groupBy("property_type")
    .agg(F.collect_set("month").alias("peak_months"))
    .orderBy("property_type")
)


## Solution 7 — Host Performance Dashboard

In [ ]:
# Full dashboard: host → properties → bookings → payments → reviews
host_props = (
    properties
    .groupBy("host_id")
    .agg(F.count("*").alias("property_count"))
)

host_booking_rev = (
    bookings
    .filter(F.col("status") == "confirmed")
    .join(properties.select("property_id", "host_id"), on="property_id")
    .groupBy("host_id")
    .agg(
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.sum(F.when(F.col("status") == "cancelled", 1).otherwise(0)).alias("cancelled_count"),
        F.count("*").alias("confirmed_bookings"),
    )
    .withColumn("cancellation_rate_pct",
        F.round(F.col("cancelled_count") / (F.col("confirmed_bookings") + F.col("cancelled_count")) * 100, 2)
    )
)

avg_guest_rating = (
    reviews.filter(F.col("is_deleted") == False)
    .join(properties.select("property_id", "host_id"), on="property_id")
    .groupBy("host_id")
    .agg(F.round(F.avg("rating"), 2).alias("avg_guest_rating"))
)

result_7 = (
    hosts
    .join(host_props,        on="host_id", how="left")
    .join(host_booking_rev,  on="host_id", how="left")
    .join(avg_guest_rating,  on="host_id", how="left")
    .select(
        "host_id", "name", "total_revenue",
        "avg_guest_rating", "cancellation_rate_pct"
    )
    .orderBy(F.col("total_revenue").desc())
)
